# CS778 Assignment 2 — Parallel Autoregressive Text Generation with a KV Cache

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` (CPU, float32)

This notebook implements the generation loop by hand, starting from the raw forward pass `model(...)`.
It does **not** use `model.generate()`, `pipeline()`, `prepare_inputs_for_generation` or any other `GenerationMixin` method.
The only model call is `model(input_ids=..., attention_mask=..., position_ids=..., past_key_values=..., use_cache=...)`, which returns the logits and the KV cache.

Contents
1. Setup and model loading
2. Prompts and batch tokenization (left padding + attention mask)
3. Token selection: greedy and temperature sampling
4. The parallel generation loop (with and without the KV cache)
5. Task 1 outputs: greedy and temperature sampling
6. Task 2 experiments: input shapes, cache on vs off, one-by-one vs batched
7. Summary table

## 1. Setup and model loading

In [1]:
import time
import warnings
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore", message="IProgress not found")
transformers.utils.logging.disable_progress_bar()

torch.manual_seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("CPU threads used by torch:", torch.get_num_threads())

/home/ashik/cs778_ass_2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cpu | transformers 5.21.0
CPU threads used by torch: 8


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 128

t0 = time.perf_counter()
# Left padding: every prompt ends at the same (last) column, so the next token
# of every row is predicted from position -1 of the logits.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.eval()
print(f"Model loaded in {time.perf_counter() - t0:.1f} s (not included in any timing below)")

# Qwen2.5-Instruct ends an answer with <|im_end|>; <|endoftext|> is also an end token.
EOS_IDS = torch.tensor(sorted({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|endoftext|>")}))
PAD_ID = tokenizer.pad_token_id
print("EOS ids:", EOS_IDS.tolist(), [tokenizer.decode([i]) for i in EOS_IDS.tolist()])
print("PAD id :", PAD_ID, repr(tokenizer.pad_token))
print("Vocabulary size (logits width):", model.config.vocab_size)

Model loaded in 4.3 s (not included in any timing below)
EOS ids: [151643, 151645] ['<|endoftext|>', '<|im_end|>']
PAD id : 151643 '<|endoftext|>'
Vocabulary size (logits width): 151936


## 2. Prompts and batch tokenization

Four prompts of clearly different lengths. Each one is wrapped in the model's chat template, and then the four are tokenized **as one batch**.
Because they have different lengths, the shorter ones are **left-padded**, and the attention mask marks real tokens with `1` and padding with `0`.

In [3]:
PROMPTS = [
    "What is 2+2?",
    "Name three primary colors.",
    "Explain in two sentences why the sky looks blue during the day.",
    "I am planning a weekend trip to the mountains with two friends. We will hike during the day "
    "and camp at night. Give me a short packing list of the most important items we should bring.",
]

def build_chat_texts(prompts):
    return [
        tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        for p in prompts
    ]

def tokenize_batch(prompts):
    return tokenizer(build_chat_texts(prompts), return_tensors="pt", padding=True)

batch = tokenize_batch(PROMPTS)
print("input_ids shape     :", tuple(batch["input_ids"].shape))
print("attention_mask shape:", tuple(batch["attention_mask"].shape))
for i, p in enumerate(PROMPTS):
    n_real = int(batch["attention_mask"][i].sum())
    n_pad = batch["input_ids"].shape[1] - n_real
    print(f"prompt {i}: {len(p.split()):3d} words -> {n_real:3d} tokens, {n_pad:3d} pad tokens on the left")

input_ids shape     : (4, 68)
attention_mask shape: (4, 68)
prompt 0:   3 words ->  36 tokens,  32 pad tokens on the left
prompt 1:   4 words ->  34 tokens,  34 pad tokens on the left
prompt 2:  12 words ->  43 tokens,  25 pad tokens on the left
prompt 3:  36 words ->  68 tokens,   0 pad tokens on the left


In [4]:
# A look at the left edge of the padded batch: pads (id 151643) and mask zeros sit on the left.
print("first 12 columns of input_ids:\n", batch["input_ids"][:, :12])
print("first 12 columns of attention_mask:\n", batch["attention_mask"][:, :12])
print("last 4 columns (every row ends with '<|im_start|>assistant\\n'):\n", batch["input_ids"][:, -4:])

first 12 columns of input_ids:
 tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643],
        [151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643],
        [151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643],
        [151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817]])
first 12 columns of attention_mask:
 tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
last 4 columns (every row ends with '<|im_start|>assistant\n'):
 tensor([[   198, 151644,  77091,    198],
        [   198, 151644,  77091,    198],
        [   198, 151644,  77091,    198],
        [   198, 151644,  77091,    198]])


## 3. Choosing the next token

The model returns logits of shape `(batch, seq_len, vocab)`. Only the last position matters for the next token, so the loop takes `logits[:, -1, :]`, which has shape `(batch, vocab)`.

* **Greedy:** `argmax` over the vocabulary.
* **Temperature sampling:** divide the logits by `T`, apply `softmax` to get probabilities, then draw one token with `torch.multinomial`. `T < 1` sharpens the distribution and `T > 1` flattens it.

In [5]:
def choose_next_token(last_logits, mode="greedy", temperature=1.0, generator=None):
    # last_logits: (batch, vocab) -> returns (batch,) token ids
    if mode == "greedy":
        return torch.argmax(last_logits, dim=-1)
    if mode == "sample":
        if temperature <= 0:
            raise ValueError("temperature must be > 0 for sampling")
        probs = torch.softmax(last_logits / temperature, dim=-1)
        return torch.multinomial(probs, num_samples=1, generator=generator).squeeze(-1)
    raise ValueError(f"unknown mode {mode!r}")

## 4. The parallel generation loop

`generate_batch` is the whole decoder. At every step it does **one** forward pass over **all** prompts in the batch.

* **Position ids.** With left padding, the real tokens of a short prompt do not start at column 0. Positions are therefore computed from the mask as `cumsum(mask) - 1`, so every prompt's first real token gets position 0. Pad columns get a dummy position, and the mask hides them anyway.
* **KV cache on (`use_cache=True`).** Step 0 (the *prefill*) sends the full padded prompts and gets back `past_key_values`. Every later step sends only the newly generated token of each row, shape `(batch, 1)`, together with the cache, the full attention mask (past + new) and the new position.
* **KV cache off (`use_cache=False`).** Every step sends the whole sequence so far (prompt + generated tokens) with no `past_key_values`, so the model recomputes keys and values for every earlier token.
* **Per-row stopping.** A row is finished once it emits an EOS token or reaches `max_new_tokens`. Finished rows stay in the batch so the tensor shapes stay rectangular. They are fed the pad token and their outputs are ignored. The loop ends when every row has finished.

In [6]:
@torch.inference_mode()
def generate_batch(prompts, mode="greedy", temperature=1.0, use_cache=True,
                   max_new_tokens=MAX_NEW_TOKENS, seed=None, record_shapes=0):
    enc = tokenize_batch(prompts)
    prompt_ids, prompt_mask = enc["input_ids"], enc["attention_mask"]
    batch_size = prompt_ids.shape[0]

    generator = None
    if seed is not None:
        generator = torch.Generator().manual_seed(seed)

    seq_ids = prompt_ids            # full sequence so far (used when the cache is off)
    attn_mask = prompt_mask         # mask over the full sequence so far
    positions = (attn_mask.cumsum(dim=-1) - 1).clamp(min=0)

    finished = torch.zeros(batch_size, dtype=torch.bool)
    new_tokens = [[] for _ in range(batch_size)]
    shapes = []                     # (input_ids shape, attention_mask shape) of the first passes
    past = None
    next_tok = None
    n_forward = 0

    for step in range(max_new_tokens):
        if use_cache and step > 0:
            # Only the newest token of every row + the cache.
            model_ids = next_tok.unsqueeze(-1)                  # (B, 1)
            model_pos = positions[:, -1:]                        # (B, 1)
        else:
            # Prefill, or cache off: the whole sequence.
            model_ids = seq_ids                                  # (B, L)
            model_pos = positions                                # (B, L)

        if len(shapes) < record_shapes:
            shapes.append((tuple(model_ids.shape), tuple(attn_mask.shape)))

        out = model(input_ids=model_ids, attention_mask=attn_mask, position_ids=model_pos,
                    past_key_values=past if use_cache else None, use_cache=use_cache)
        n_forward += 1
        if use_cache:
            past = out.past_key_values

        next_tok = choose_next_token(out.logits[:, -1, :], mode, temperature, generator)
        # Rows that already stopped just get padding.
        next_tok = torch.where(finished, torch.full_like(next_tok, PAD_ID), next_tok)

        for i in range(batch_size):
            if not finished[i]:
                new_tokens[i].append(int(next_tok[i]))
        finished |= torch.isin(next_tok, EOS_IDS)

        seq_ids = torch.cat([seq_ids, next_tok.unsqueeze(-1)], dim=-1)
        attn_mask = torch.cat([attn_mask, torch.ones(batch_size, 1, dtype=attn_mask.dtype)], dim=-1)
        positions = torch.cat([positions, positions[:, -1:] + 1], dim=-1)

        if finished.all():
            break

    texts, n_generated, stop_reason = [], [], []
    for toks in new_tokens:
        hit_eos = len(toks) > 0 and toks[-1] in EOS_IDS.tolist()
        content = toks[:-1] if hit_eos else toks
        texts.append(tokenizer.decode(content, skip_special_tokens=True).strip())
        n_generated.append(len(toks))
        stop_reason.append("EOS" if hit_eos else f"max {max_new_tokens} tokens")

    return {"texts": texts, "token_ids": new_tokens, "n_generated": n_generated,
            "stop_reason": stop_reason, "n_forward": n_forward, "shapes": shapes}


def show_results(prompts, res, title):
    print("=" * 100)
    print(title, f"| forward passes: {res['n_forward']}")
    print("=" * 100)
    for i, (p, r) in enumerate(zip(prompts, res["texts"])):
        print(f"\n[Prompt {i}] {p}")
        print(f"[Response {i}] ({res['n_generated'][i]} tokens, stopped by {res['stop_reason'][i]})\n{r}")
    print()

## 5. Task 1 — Responses (all four prompts generated in parallel, KV cache on)

### 5.1 Greedy decoding

In [7]:
greedy = generate_batch(PROMPTS, mode="greedy", use_cache=True, record_shapes=4)
show_results(PROMPTS, greedy, "GREEDY DECODING (batched, KV cache on)")

GREEDY DECODING (batched, KV cache on) | forward passes: 128

[Prompt 0] What is 2+2?
[Response 0] (8 tokens, stopped by EOS)
2+2 equals 4.

[Prompt 1] Name three primary colors.
[Response 1] (12 tokens, stopped by EOS)
Three primary colors are red, blue, and yellow.

[Prompt 2] Explain in two sentences why the sky looks blue during the day.
[Response 2] (52 tokens, stopped by EOS)
The sky appears blue during the day due to the scattering of sunlight by tiny water droplets in the Earth's atmosphere. These droplets, which are tiny ice crystals, scatter the blue light from the sun, causing it to appear as a blue hue.

[Prompt 3] I am planning a weekend trip to the mountains with two friends. We will hike during the day and camp at night. Give me a short packing list of the most important items we should bring.
[Response 3] (128 tokens, stopped by max 128 tokens)
Certainly! Here's a packing list for your weekend trip to the mountains with two friends:

### Essential Items:
1. **Hiking Gea

### 5.2 Temperature sampling (T = 0.7, fixed seed for reproducibility)

In [8]:
TEMPERATURE = 0.7
sampled = generate_batch(PROMPTS, mode="sample", temperature=TEMPERATURE, use_cache=True, seed=1234)
show_results(PROMPTS, sampled, f"TEMPERATURE SAMPLING, T = {TEMPERATURE} (batched, KV cache on)")

TEMPERATURE SAMPLING, T = 0.7 (batched, KV cache on) | forward passes: 128

[Prompt 0] What is 2+2?
[Response 0] (9 tokens, stopped by EOS)
2 + 2 equals 4.

[Prompt 1] Name three primary colors.
[Response 1] (54 tokens, stopped by EOS)
Three primary colors are red, yellow, and blue. These colors are fundamental in the visual spectrum and are used to create all other colors in the rainbow and beyond. Each primary color has unique properties that make them ideal for specific effects and applications in design and technology.

[Prompt 2] Explain in two sentences why the sky looks blue during the day.
[Response 2] (61 tokens, stopped by EOS)
The sky appears blue during the day due to the scattering of sunlight by water droplets in the atmosphere. This is caused by the refraction and reflection of sunlight as it passes through the Earth's atmosphere, and the way these changes in light travel through the air and reflect off objects on the surface.

[Prompt 3] I am planning a weekend trip to 

## 6. Task 2 — Experiments

All runs use **greedy decoding** and the same four prompts. Model loading is not timed. Each timed configuration is run once first as a warm-up, then timed over `N_REPEATS` runs with `time.perf_counter()`, and the mean is reported.

### 6.1 Shapes passed to the model in the first four forward passes (Task 2.3)

In [9]:
def print_shapes(shapes, title):
    print(title)
    print(f"{'pass':>11} | {'input_ids':>12} | {'attention_mask':>14}")
    for k, (ids_shape, mask_shape) in enumerate(shapes):
        label = "1 (prefill)" if k == 0 else str(k + 1)
        print(f"{label:>11} | {str(ids_shape):>12} | {str(mask_shape):>14}")

print_shapes(greedy["shapes"], "KV cache ON (the cache is used from the 2nd pass onwards):")
print()
cache_off_preview = generate_batch(PROMPTS, mode="greedy", use_cache=False, max_new_tokens=4, record_shapes=4)
print_shapes(cache_off_preview["shapes"], "KV cache OFF (for comparison: the whole sequence is re-sent every pass):")

KV cache ON (the cache is used from the 2nd pass onwards):
       pass |    input_ids | attention_mask
1 (prefill) |      (4, 68) |        (4, 68)
          2 |       (4, 1) |        (4, 69)
          3 |       (4, 1) |        (4, 70)
          4 |       (4, 1) |        (4, 71)



KV cache OFF (for comparison: the whole sequence is re-sent every pass):
       pass |    input_ids | attention_mask
1 (prefill) |      (4, 68) |        (4, 68)
          2 |      (4, 69) |        (4, 69)
          3 |      (4, 70) |        (4, 70)
          4 |      (4, 71) |        (4, 71)


### 6.2 KV cache on vs off (Task 2.1)

In [10]:
N_REPEATS = 3

def time_runs(fn, repeats=N_REPEATS):
    fn()  # warm-up, not timed
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - t0)
    return sum(times) / len(times), times, result

t_cache_on, runs_on, res_on = time_runs(lambda: generate_batch(PROMPTS, mode="greedy", use_cache=True))
t_cache_off, runs_off, res_off = time_runs(lambda: generate_batch(PROMPTS, mode="greedy", use_cache=False))

print(f"KV cache ON : {t_cache_on:7.2f} s  (runs: {', '.join(f'{t:.2f}' for t in runs_on)})")
print(f"KV cache OFF: {t_cache_off:7.2f} s  (runs: {', '.join(f'{t:.2f}' for t in runs_off)})")
print(f"Speedup from the KV cache: {t_cache_off / t_cache_on:.2f}x")
print(f"Forward passes per run: {res_on['n_forward']} (on) vs {res_off['n_forward']} (off)")
print("Tokens generated per prompt:", res_on["n_generated"])
same = res_on["token_ids"] == res_off["token_ids"]
print("Greedy outputs identical with cache on and off:", same)
if not same:
    for i, (a, b) in enumerate(zip(res_on["token_ids"], res_off["token_ids"])):
        d = next((k for k, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
        print(f"  prompt {i}: identical for the first {d} tokens (float rounding can flip a near-tie)")

KV cache ON :   27.22 s  (runs: 27.18, 27.24, 27.25)
KV cache OFF:  276.74 s  (runs: 281.70, 276.10, 272.43)
Speedup from the KV cache: 10.17x
Forward passes per run: 128 (on) vs 128 (off)
Tokens generated per prompt: [8, 12, 52, 128]
Greedy outputs identical with cache on and off: True


### 6.3 One by one vs one batch (Task 2.2)

In [11]:
def run_one_by_one():
    # Four separate loops, each with a batch of one prompt (KV cache on, greedy).
    return [generate_batch([p], mode="greedy", use_cache=True) for p in PROMPTS]

def run_batched():
    return generate_batch(PROMPTS, mode="greedy", use_cache=True)

t_seq, runs_seq, res_seq = time_runs(run_one_by_one)
t_bat, runs_bat, res_bat = time_runs(run_batched)

print(f"One by one (4 loops of batch 1): {t_seq:7.2f} s  (runs: {', '.join(f'{t:.2f}' for t in runs_seq)})")
print(f"Batched (1 loop of batch 4)    : {t_bat:7.2f} s  (runs: {', '.join(f'{t:.2f}' for t in runs_bat)})")
print(f"Speedup from batching: {t_seq / t_bat:.2f}x")
print(f"Forward passes: {sum(r['n_forward'] for r in res_seq)} (one by one, total) vs {res_bat['n_forward']} (batched)")
for i, r in enumerate(res_seq):
    same = r["token_ids"][0] == res_bat["token_ids"][i]
    print(f"  prompt {i}: {r['n_generated'][0]:3d} tokens alone, {res_bat['n_generated'][i]:3d} in batch, identical output: {same}")

One by one (4 loops of batch 1):   23.27 s  (runs: 21.56, 24.14, 24.10)
Batched (1 loop of batch 4)    :   30.58 s  (runs: 30.08, 29.11, 32.55)
Speedup from batching: 0.76x
Forward passes: 200 (one by one, total) vs 128 (batched)
  prompt 0:   8 tokens alone,   8 in batch, identical output: True
  prompt 1:  12 tokens alone,  12 in batch, identical output: True
  prompt 2:  52 tokens alone,  52 in batch, identical output: True
  prompt 3: 128 tokens alone, 128 in batch, identical output: True


### 6.4 Where the batched time goes

The two timings above are reduced to the cost of a **single forward pass**, so batch-1 and batch-4 steps can be compared directly. *Useful row-steps* are steps where a row is still generating. *Wasted row-steps* are steps where the row has already stopped but still sits in the batch.

In [12]:
seq_passes = sum(r["n_forward"] for r in res_seq)
bat_passes = res_bat["n_forward"]
ms_per_pass_b1 = 1000 * t_seq / seq_passes
ms_per_pass_b4 = 1000 * t_bat / bat_passes
useful = sum(res_bat["n_generated"])
total_row_steps = len(PROMPTS) * bat_passes

print(f"one by one: {seq_passes:4d} passes of batch 1 -> {ms_per_pass_b1:6.1f} ms per pass")
print(f"batched   : {bat_passes:4d} passes of batch 4 -> {ms_per_pass_b4:6.1f} ms per pass")
print(f"cost of a batch-4 pass relative to a batch-1 pass: {ms_per_pass_b4 / ms_per_pass_b1:.2f}x (4 rows for the price of {ms_per_pass_b4 / ms_per_pass_b1:.2f})")
print(f"throughput if every row were busy: batch 1 = {1000 / ms_per_pass_b1:5.1f} tok/s, batch 4 = {4000 / ms_per_pass_b4:5.1f} tok/s")
print(f"batched run: {useful} useful row-steps out of {total_row_steps} -> {total_row_steps - useful} wasted ({100 * (total_row_steps - useful) / total_row_steps:.0f}%)")
print(f"answer lengths {res_bat['n_generated']}: the batch keeps all four rows until the longest one finishes")

one by one:  200 passes of batch 1 ->  116.3 ms per pass
batched   :  128 passes of batch 4 ->  238.9 ms per pass
cost of a batch-4 pass relative to a batch-1 pass: 2.05x (4 rows for the price of 2.05)
throughput if every row were busy: batch 1 =   8.6 tok/s, batch 4 =  16.7 tok/s
batched run: 200 useful row-steps out of 512 -> 312 wasted (61%)
answer lengths [8, 12, 52, 128]: the batch keeps all four rows until the longest one finishes


## 7. Summary of timings

In [13]:
rows = [
    ("KV cache ON  (batched, greedy)", t_cache_on),
    ("KV cache OFF (batched, greedy)", t_cache_off),
    ("One by one   (cache on, greedy)", t_seq),
    ("Batched      (cache on, greedy)", t_bat),
]
print(f"{'Configuration':<34} | {'mean time (s)':>13}")
print("-" * 52)
for name, t in rows:
    print(f"{name:<34} | {t:13.2f}")
print("-" * 52)
print(f"{'Speedup: cache on vs off':<34} | {t_cache_off / t_cache_on:12.2f}x")
print(f"{'Speedup: batched vs one by one':<34} | {t_seq / t_bat:12.2f}x")
print(f"\nmax_new_tokens = {MAX_NEW_TOKENS}, mean of {N_REPEATS} runs after a warm-up, CPU threads = {torch.get_num_threads()}")

Configuration                      | mean time (s)
----------------------------------------------------
KV cache ON  (batched, greedy)     |         27.22
KV cache OFF (batched, greedy)     |        276.74
One by one   (cache on, greedy)    |         23.27
Batched      (cache on, greedy)    |         30.58
----------------------------------------------------
Speedup: cache on vs off           |        10.17x
Speedup: batched vs one by one     |         0.76x

max_new_tokens = 128, mean of 3 runs after a warm-up, CPU threads = 8


**Observations**

* **KV cache.** Without the cache, step *t* re-runs the model over all `L + t` tokens, so the total work grows roughly quadratically with the output length. With the cache, every step after the prefill processes a single new token per row and reuses the stored keys and values of earlier tokens. The shape table in 6.1 shows this directly: `input_ids` is `(4, L)` on the prefill and `(4, 1)` afterwards with the cache, while without the cache it grows to `(4, L+1)`, `(4, L+2)`, and so on.
* **Batching.** In a decoding step, every weight matrix of the model must be read from memory once, however many rows are in the batch. A batch of 4 shares that one read across four prompts, so a batch-4 pass costs much less than four batch-1 passes (section 6.4 measures the ratio). That raises the number of tokens generated per second.
* **Why the batched total can still lose.** The batch keeps every row until the *longest* answer is done. Here the greedy answers are 8, 12, 52 and 128 tokens long, so most row-steps in the batched run are spent on rows that have already stopped. One by one, each prompt runs only for as many steps as it needs (200 passes in total). On this laptop CPU (fp32, 8 threads, AVX2 only), a batch-4 pass costs close to two batch-1 passes, because the matrix multiplications for 4 rows are already partly compute-bound. Together these make one-by-one slightly faster *for this particular set of answers*. When the answers are of similar length, or with more rows per batch, batching wins (each row-step is about twice as cheap here). On a GPU the gain is much larger, because a batch-4 pass costs almost the same as a batch-1 pass.